In [1]:
import sys
sys.path.insert(0, "src")
from kai.data.grb211211a_loaders import (
    load_xrt_lightcurve, load_xrt_as_sed,
    load_optical_photometry, summarise_xrt
)

In [3]:
# Load XRT
df_xrt = load_xrt_lightcurve(
    "../data/GRB211211A/xrt_flux_lightcurve.qdp"
)

# Check Epoch 1: t ~ 0.04 days = 3456 s
print("=== Epoch 1 (t ~ 0.04 days) ===")
epoch1_xrt = df_xrt[
    (df_xrt['t_s'] >= 2000) & (df_xrt['t_s'] <= 5000)
]
print(f"XRT points: {len(epoch1_xrt)}")
print(epoch1_xrt[['t_s', 'flux', 'flux_pos', 'mode']].to_string())

print("\n=== Epoch 2 (t ~ 1.35 days) ===")
epoch2_xrt = df_xrt[
    (df_xrt['t_s'] >= 1e5) & (df_xrt['t_s'] <= 1.5e5)
]
print(f"XRT points: {len(epoch2_xrt)}")
print(epoch2_xrt[['t_s', 'flux', 'flux_pos', 'mode']].to_string())

print("\n=== Optical data per epoch ===")
df_opt = load_optical_photometry()
for t in [0.043, 0.208, 0.417, 1.354, 4.188]:
    sub = df_opt[abs(df_opt['t_days'] - t) < 0.01]
    if len(sub):
        print(f"t={t:.3f} days: {len(sub)} pts in {sorted(sub['band'].unique())}")

=== Epoch 1 (t ~ 0.04 days) ===
XRT points: 28
          t_s          flux      flux_pos mode
233  3519.350  5.162925e-11  1.168388e-11   PC
234  3594.076  5.079186e-11  1.147709e-11   PC
235  3643.923  9.059108e-11  1.898794e-11   PC
236  3693.282  5.338301e-11  1.185087e-11   PC
237  3748.477  1.071334e-10  2.251384e-11   PC
238  3798.858  5.713457e-11  1.253895e-11   PC
239  3848.596  6.779238e-11  1.524971e-11   PC
240  3908.497  5.881387e-11  1.321022e-11   PC
241  3952.913  9.044079e-11  2.034441e-11   PC
242  4005.533  5.121631e-11  1.162548e-11   PC
243  4061.210  6.058765e-11  1.366999e-11   PC
244  4134.868  4.098932e-11  9.262074e-12   PC
245  4195.365  8.460484e-11  1.903162e-11   PC
246  4239.748  6.846464e-11  1.544723e-11   PC
247  4297.936  5.951252e-11  1.313578e-11   PC
248  4354.639  5.876050e-11  1.298844e-11   PC
249  4411.248  7.044866e-11  1.591879e-11   PC
250  4469.859  5.514755e-11  1.210287e-11   PC
251  4520.865  8.913575e-11  1.953416e-11   PC
252  4561.064

In [4]:
# Check full XRT time coverage
print("Full XRT time coverage:")
print(df_xrt.groupby('mode')['t_s'].agg(['min', 'max', 'count']))

# Check for any XRT data between 0.5 and 2 days
mask = (df_xrt['t_s'] >= 0.5*86400) & (df_xrt['t_s'] <= 2*86400)
print(f"\nXRT points between 0.5-2 days: {mask.sum()}")
print(df_xrt[mask][['t_s', 'flux', 'mode']])

# Check what's between 1e4 and 2e5 s
mask2 = (df_xrt['t_s'] >= 1e4) & (df_xrt['t_s'] <= 2e5)
print(f"\nXRT points between 1e4-2e5 s: {mask2.sum()}")
print(df_xrt[mask2][['t_s', 'flux', 'mode']].head(20))

Full XRT time coverage:
              min           max  count
mode                                  
PC    3519.350000  1.208832e+07     53
WT      69.935415  2.950810e+02    233

XRT points between 0.5-2 days: 6
           t_s          flux mode
279  56514.699  1.807056e-12   PC
280  56997.160  1.744461e-12   PC
281  62273.158  1.198512e-12   PC
282  67663.355  1.456404e-12   PC
283  68100.424  2.003581e-12   PC
284  73657.465  1.192120e-12   PC

XRT points between 1e4-2e5 s: 24
           t_s          flux mode
261  15975.162  1.232620e-11   PC
262  16050.391  1.623776e-11   PC
263  16155.276  1.463131e-11   PC
264  16312.729  1.567342e-11   PC
265  16435.371  2.202549e-11   PC
266  16566.378  1.161535e-11   PC
267  16652.278  1.477422e-11   PC
268  16752.025  8.676823e-12   PC
269  16874.426  1.581660e-11   PC
270  16951.153  1.579304e-11   PC
271  17028.997  1.388837e-11   PC
272  17129.919  1.372096e-11   PC
273  17232.839  1.025436e-11   PC
274  17365.285  1.161401e-11   PC
275 

In [5]:
# Check optical data near 0.7 days
df_opt = load_optical_photometry()
print("Optical epochs available (days):")
print(sorted(df_opt['t_days'].unique()))

# XRT data at ~0.7 days
mask = (df_xrt['t_s'] >= 5e4) & (df_xrt['t_s'] <= 9e4)
epoch3_xrt = df_xrt[mask]
print(f"\nXRT at t~0.7 days: {len(epoch3_xrt)} points")
print(epoch3_xrt[['t_s','flux','flux_pos','mode']])

# Average XRT flux at ~0.7 days
t_center = 65000  # s
dt       = 15000  # s
mask2 = (df_xrt['t_s'] >= t_center-dt) & (df_xrt['t_s'] <= t_center+dt)
sub = df_xrt[mask2]
flux_avg = sub['flux'].mean()
flux_err = sub['flux_pos'].mean()
print(f"\nAverage XRT flux at t~{t_center}s: {flux_avg:.3e} ± {flux_err:.3e} erg/cm²/s")

Optical epochs available (days):
[np.float64(0.043), np.float64(0.208), np.float64(0.417), np.float64(1.354), np.float64(4.188)]

XRT at t~0.7 days: 6 points
           t_s          flux      flux_pos mode
279  56514.699  1.807056e-12  4.739320e-13   PC
280  56997.160  1.744461e-12  4.185159e-13   PC
281  62273.158  1.198512e-12  2.539373e-13   PC
282  67663.355  1.456404e-12  3.829782e-13   PC
283  68100.424  2.003581e-12  4.637976e-13   PC
284  73657.465  1.192120e-12  2.524578e-13   PC

Average XRT flux at t~65000s: 1.567e-12 ± 3.743e-13 erg/cm²/s


In [6]:
for t_d, label in [(0.208, "Epoch 2"), (0.417, "Epoch 3")]:
    t_s = t_d * 86400
    dt  = 0.1 * 86400
    mask = (df_xrt['t_s'] >= t_s-dt) & (df_xrt['t_s'] <= t_s+dt)
    sub  = df_xrt[mask]
    print(f"\n{label} (t~{t_d}d = {t_s:.0f}s):")
    print(f"  XRT points: {len(sub)}")
    if len(sub):
        print(f"  flux: {sub['flux'].mean():.3e} ± "
              f"{sub['flux_pos'].mean():.3e} erg/cm2/s")
        print(sub[['t_s','flux','mode']].to_string())


Epoch 2 (t~0.208d = 17971s):
  XRT points: 18
  flux: 1.306e-11 ± 2.916e-12 erg/cm2/s
           t_s          flux mode
261  15975.162  1.232620e-11   PC
262  16050.391  1.623776e-11   PC
263  16155.276  1.463131e-11   PC
264  16312.729  1.567342e-11   PC
265  16435.371  2.202549e-11   PC
266  16566.378  1.161535e-11   PC
267  16652.278  1.477422e-11   PC
268  16752.025  8.676823e-12   PC
269  16874.426  1.581660e-11   PC
270  16951.153  1.579304e-11   PC
271  17028.997  1.388837e-11   PC
272  17129.919  1.372096e-11   PC
273  17232.839  1.025436e-11   PC
274  17365.285  1.161401e-11   PC
275  21741.539  8.324836e-12   PC
276  21855.286  1.024818e-11   PC
277  21952.826  1.223642e-11   PC
278  22089.335  7.221464e-12   PC

Epoch 3 (t~0.417d = 36029s):
  XRT points: 0
